## Steady-state Single Variable Analysis

## Note

- Must rerun for final analysis, since the control bootstrapping output a potentially missing file

## Notes on current version of this code

- For now, taking the following data types into account for clustering:
    - Mean
    - CV
    - Pearson Correlations (bivariate relations)
    - ACFs (not sure yet - estimation needs to be worked out)

- In this notebook, I will attempt to settle on a good set of estimators for the various quantities of interest
- Note that this does **not** cover the issue of data transformation and normalization, which applies to the clustering domain
- This notebook will only help me determine the best (minimum variance) estimators, p-values and confidence intervals

### Outstanding Issue 1

- The superiority of the Qn estimator for ACF estimation has not been demonstrated
- It has higher variance than the standard pearson estimator in this case, but it is possible that it is significantly more biased
- To measure the MSE, I need access to true ACF values
- Without a ground truth measurement of the ACF, not sure how to do this at the moment
- Still going to measure Qn for the ACF even without this guarentee

### Outstanding Issue 2

- Have not yet replaced the standard CV estimate with a robust estimator of scale (i.e. Qn)

In [ ]:
import paulssonlab.deaton.trenchripper.trenchripper as tr

import numpy as np
import pandas as pd
import seaborn as sns
import scipy as sp
import scipy.stats
from scipy.stats import ttest_ind

import sklearn as skl
import dask.dataframe as dd
import dask.array as da
import dask
import warnings
import copy
import random
import shutil
import os
from sklearn.metrics.pairwise import euclidean_distances,manhattan_distances,cosine_distances
from matplotlib.lines import Line2D

from sklearn.metrics import silhouette_score
import statsmodels.stats.multitest
from sklearn.linear_model import LinearRegression
from sklearn.cluster import AffinityPropagation
from sklearn.manifold import TSNE
from sklearn.cluster import AgglomerativeClustering
from dask.distributed import wait

from matplotlib import pyplot as plt
import ast

from recombinator.block_bootstrap import stationary_bootstrap
from recombinator.tapered_block_bootstrap import tapered_block_bootstrap
from statsmodels.robust.scale import qn_scale
import pandas as pd
import xarray as xr

import pylab
import scipy.cluster.hierarchy as sch

import matplotlib.gridspec as gridspec
import matplotlib as mpl

import holoviews as hv
hv.extension('bokeh')
random.seed(42)
np.random.seed(42)

warnings.filterwarnings(action='once',category=UserWarning)

### Initial Data Processing

Here, I am going to try and replicate (to some extant) the corrections from "Genomewide phenotypic analysis of growth, cell morphogenesis, and cell cycle events in Escherichia coli"

#### Start Dask

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="02:00:00",
    local=False,
    n_workers=40,
    n_workers_min=40,
    memory="16GB",
    working_directory="/home/lag36/scratch/lag36/dask/",
)
dask_controller.startdask()

In [ ]:
dask_controller.daskclient

## Cell Cycle Analysis

- Preliminary processing taken from the main clustering ipynb
- After this notebook is completed, merge with that notebook for a complete pipeline

#### Note

- Need to accomadate a simple bootstrap (later)

### Import Necessary Dataframes

In [ ]:
cell_cycle_df = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Lineage_Cell_Cycle_Merged/", engine="pyarrow",calculate_divisions=True)
timepoints_df = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Lineage_Observations_Merged/", engine="pyarrow",calculate_divisions=True)
growth_df = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Lineage_Growth_Observations_Merged/", engine="pyarrow",calculate_divisions=True)

In [ ]:
growth_df.columns

#### Select Parameters to Analyze

In [ ]:
cell_cycle_columns = ['Multi-Experiment Global CellID', 'Multi-Experiment FOV', 'Global CellID', 'phenotype trenchid', 'File Parquet Index', 'fov',\
       'row', 'trench', 'initial timepoints', 'Mother', 'CellID', 'Mother CellID', 'Daughter CellID 1', 'Daughter CellID 2', 'Sister CellID',\
                      'Experiment #', 'Birth: major_axis_length', 'Division: major_axis_length',\
                      'Delta: major_axis_length', 'Septum Position Length Normalized', 'Birth: Volume', 'Division: Volume', 'Delta: Volume',\
                      'Final time (s)','Initial time (s)','Delta time (s)','Growth Rate: major_axis_length','Growth Rate: Volume',\
                     'Birth: Number of Nucleoids', 'Division: Number of Nucleoids', 'Birth: Total Nucleoid Area', 'Division: Total Nucleoid Area' ] # New parameters


timepoints_columns = ['Global CellID-Cell Cycle timepoints', 'phenotype trenchid', 'Global CellID', 'Observation time (s)', 'major_axis_length',\
                     'minor_axis_length', 'YFP mean_intensity', 'Experiment #', 'Multi-Experiment Global CellID',\
                     'Number of Nucleoids', 'Total Nucleoid Area'] # NEW PARAMETERS


growth_columns = ['Global CellID-Growth Rate Measurement Index', 'phenotype trenchid', 'Global CellID', 'Measurement time (s)',\
                  'Instantaneous Growth Rate: major_axis_length', 'Instantaneous Growth Rate: Volume', 'Experiment #', 'Multi-Experiment Global CellID']

cell_cycle_df = cell_cycle_df[cell_cycle_columns]
timepoints_df = timepoints_df[timepoints_columns]
growth_df = growth_df[growth_columns]

#### Extract all cell cycle data from dataframes (defunct)

In [ ]:
# parameters_to_avg = ["minor_axis_length","mCherry mean_intensity"] #other parameters already captured elsewhere for cell cycle

# cellid_groupby = timepoints_df.groupby("Multi-Experiment Global CellID")
# cellid_groupby = cellid_groupby[parameters_to_avg].mean()
# avg_over_cell_cycle = cellid_groupby.compute()
# avg_over_cell_cycle = avg_over_cell_cycle.rename(dict(zip(parameters_to_avg,["Mean: " + item for item in parameters_to_avg])), axis=1)

# final_cell_cycle_df = cell_cycle_df.reset_index().set_index("Multi-Experiment Global CellID",sorted=True)
# final_cell_cycle_df = final_cell_cycle_df.join(avg_over_cell_cycle)
# final_cell_cycle_df = final_cell_cycle_df.reset_index().set_index("Multi-Experiment Phenotype Trenchid",sorted=True)

### Classify proliferating cells
- Classify trenches as proliferating or non-proliferating and exclude from subsequent analysis where appropriate
- Proliferation criterion:
    - Median instantanous growth rate for the trench must give a doubling time of at least once every 2 hours, that is $ \lambda > \frac{\ln(2)}{3} \frac{1}{hr} $ 
    - Median doubling time (for reported cell cycles) must give a doubling time of at least once every 2 hours
    - The number of cell cycle reports (of the mother cell only) must also average out to once every 2 hours

In [ ]:
steady_state_time = 7*60*60 #arbitrary definition for now (seconds) 7 hours
final_time = 10*60*60 # end of the experiment
steady_state_interval = final_time-steady_state_time

doubling_threshold = 2
growth_rate_threshold = np.log(2)/doubling_threshold
doubling_time_threshold = doubling_threshold*3600
n_mother_cycles = int(np.ceil(steady_state_interval/doubling_time_threshold))

In [ ]:
median_growth_rate_series = growth_df.groupby("Multi-Experiment Phenotype Trenchid")["Instantaneous Growth Rate: major_axis_length"].apply(lambda x: np.nanmedian(x)).compute()
median_doubling_time_series = cell_cycle_df.groupby("Multi-Experiment Phenotype Trenchid")["Delta time (s)"].apply(lambda x: np.nanmedian(x)).compute()
n_cell_cycles_series = cell_cycle_df[cell_cycle_df["Mother"]==True].groupby("Multi-Experiment Phenotype Trenchid").size().compute()

In [ ]:
trenches_above_growth_thr = median_growth_rate_series>growth_rate_threshold
trenches_below_doubling_thr = median_doubling_time_series<doubling_time_threshold
trenches_above_n_cycles_thr = n_cell_cycles_series>n_mother_cycles
proliferating_trench_df = pd.concat([trenches_above_growth_thr,trenches_below_doubling_thr,trenches_above_n_cycles_thr],join="inner",axis=1)
proliferating_trench_df["Proliferating"] = np.all(proliferating_trench_df.values, axis=1)
proliferating_trench_df = proliferating_trench_df.drop(["Instantaneous Growth Rate: major_axis_length","Delta time (s)",0], axis=1)
cell_cycle_df = cell_cycle_df.join(proliferating_trench_df,how="inner").persist()
wait(cell_cycle_df);

In [ ]:
plt.hist(median_growth_rate_series[median_growth_rate_series>growth_rate_threshold],bins=50,log=True,range=(0,2))
plt.hist(median_growth_rate_series[median_growth_rate_series<=growth_rate_threshold],bins=50,log=True,range=(0,2))
plt.xlabel("Growth Rate (1/hr)",fontsize=14)
plt.show()

plt.hist(median_doubling_time_series[median_doubling_time_series<=doubling_time_threshold]/3600,bins=20,log=True,range=(0,4))
plt.hist(median_doubling_time_series[median_doubling_time_series>doubling_time_threshold]/3600,bins=20,log=True,range=(0,4))
plt.xlabel("Doubling Time (hr)",fontsize=14)
plt.show()

plt.hist(n_cell_cycles_series[n_cell_cycles_series>n_mother_cycles],bins=40,log=True,range=(0,40))
plt.hist(n_cell_cycles_series[n_cell_cycles_series<=n_mother_cycles],bins=40,log=True,range=(0,40))

plt.xlabel("N Mother Cell Cycles",fontsize=14)
plt.show()

#### Add sgRNA info for real data

In [ ]:
final_output_df_barcodes = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Final_Barcodes_df_Merged",engine="pyarrow")
final_output_df_barcodes = final_output_df_barcodes.set_index("Multi-Experiment Phenotype Trenchid",sorted=True).groupby("Multi-Experiment Phenotype Trenchid",sort=False).apply(lambda x: x.iloc[0])
final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index("oDEPool7_id",drop=True)
final_output_df_barcodes = final_output_df_barcodes.reset_index().set_index("Multi-Experiment Phenotype Trenchid").persist()
final_output_df_barcodes = final_output_df_barcodes[['oDEPool7_id','Barcode','sgRNA','Closest Hamming Distance', 'EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID', 'barcodeid']]
wait(final_output_df_barcodes);
cell_cycle_df = cell_cycle_df.join(final_output_df_barcodes).persist()
timepoints_df = timepoints_df.join(final_output_df_barcodes).persist()
growth_df = growth_df.join(final_output_df_barcodes).persist()
wait(cell_cycle_df);

dask_controller.daskclient.cancel(final_output_df_barcodes)

In [ ]:
cell_cycle_df_only_proliferating = cell_cycle_df[cell_cycle_df["Proliferating"]]

n_obs = cell_cycle_df.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs = pd.DataFrame(n_obs).rename({0:"N Observations"}, axis=1).sort_index()
n_obs_prolif = cell_cycle_df_only_proliferating.groupby("oDEPool7_id",sort=False).apply(lambda x: len(x.index.unique()), meta=int).compute()
n_obs_prolif = pd.DataFrame(n_obs_prolif).rename({0:"N Observations Proliferating"}, axis=1).sort_index()
obs_df = pd.concat([n_obs,n_obs_prolif],axis=1)
obs_df.loc[np.isnan(obs_df["N Observations Proliferating"]),"N Observations Proliferating"] = 0
obs_df["N Observations Proliferating"] = obs_df["N Observations Proliferating"].astype(int)

cell_cycle_df = cell_cycle_df.join(obs_df,on="oDEPool7_id")

In [ ]:
#hack here to repair index issue, not sure how it happens
cell_cycle_df = cell_cycle_df.reset_index().repartition(npartitions=100).set_index("Multi-Experiment Phenotype Trenchid",sorted=True)
timepoints_df = timepoints_df.reset_index().repartition(npartitions=100).set_index("Multi-Experiment Phenotype Trenchid",sorted=True)
growth_df = growth_df.reset_index().repartition(npartitions=100).set_index("Multi-Experiment Phenotype Trenchid",sorted=True)

#### Filter for variants observed at least 10 times in the experiment

In [ ]:
## excluding this filter for now, will need to filter on a per-analysis basis
# final_cell_cycle_df = final_cell_cycle_df[final_cell_cycle_df["N Observations"]>=10]

### Determine when cells reach steady state (later) and save intermediate df as a checkpoint

In [ ]:
steady_state_time = 7*60*60#36000 #arbitrary definition for now
induction_time = 2*60*60

cell_cycle_df_ss = cell_cycle_df[cell_cycle_df["Initial time (s)"]>=steady_state_time].persist()
cell_cycle_df_preinduction = cell_cycle_df[cell_cycle_df["Final time (s)"]<=induction_time].persist()
cell_cycle_df_ss.to_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Cell_Cycle_df/", engine="pyarrow", overwrite=True)
cell_cycle_df_preinduction.to_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Cell_Cycle_df/", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(cell_cycle_df)
dask_controller.daskclient.cancel(cell_cycle_df_ss)
dask_controller.daskclient.cancel(cell_cycle_df_preinduction)

timepoints_df_ss = timepoints_df[timepoints_df["Observation time (s)"]>=steady_state_time].persist()
timepoints_df_preinduction = timepoints_df[timepoints_df["Observation time (s)"]<=induction_time].persist()
timepoints_df_ss.to_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Timepoints_df/", engine="pyarrow", overwrite=True)
timepoints_df_preinduction.to_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Timepoints_df/", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(timepoints_df)
dask_controller.daskclient.cancel(timepoints_df_ss)
dask_controller.daskclient.cancel(timepoints_df_preinduction)

growth_df_ss = growth_df[growth_df["Measurement time (s)"]>=steady_state_time].persist()
growth_df_preinduction = growth_df[growth_df["Measurement time (s)"]<=induction_time].persist()
growth_df_ss.to_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Growth_df/", engine="pyarrow", overwrite=True)
growth_df_preinduction.to_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Growth_df/", engine="pyarrow", overwrite=True)
dask_controller.daskclient.cancel(growth_df)
dask_controller.daskclient.cancel(growth_df_ss)
dask_controller.daskclient.cancel(growth_df_preinduction)
dask_controller.reset_worker_memory()

## Computing Trench-wise Estimator Distributions

- These distributions can be used to construct confidence intervals (this will help with clustering)
- They can also be re-used to compute p-values (with some modification)

### Estimators of Interest

- Mean
- Median
- CV
- MAD CV
- Pearson R
- Qn (Pearson R)

### Bootstraps of Interest

- Naive Bootstrap
- Tapered Block Bootstrap
    - Using for ACFs
- Stationary Bootstrap
    - Using for location, scale and pearson correlation (single cell cycle)

In [ ]:
### some helper functions for the various estimators

def coeff_var(x,axis=None):
    return np.std(x,axis=axis)/np.mean(x,axis=axis)

def nancoeff_var(x,axis=None):
    return np.nanstd(x,axis=axis)/np.nanmean(x,axis=axis)

def nancoeff_var_mad(x,axis=None,coeff=1/sp.stats.norm().ppf(3/4)):
    # median uses quickselect
    median = np.nanmedian(x,axis=axis,keepdims=True)
    abs_deviation = np.abs(x-median) #broadcast the median subtraction
    mad = np.nanmedian(abs_deviation,axis=axis,keepdims=True)
    mad_cv = coeff*(mad/median)
    mad_cv = np.squeeze(mad_cv, axis=axis)
    return mad_cv

def nanpearsonr(x):
    nanmask = np.any(np.isnan(x),axis=0)
    x_nonan = x[:,~nanmask]
    r = np.corrcoef(x_nonan)[0,1]
    return r

def dn_fn(n):
    dn_list = np.array([np.NaN,np.NaN,0.399,0.994,0.512,0.844,0.611,0.857,0.669,0.872])
    dn = np.zeros(n.shape)
    under_10_mask = (n<=9)
    even_mask = (n%2==0)
    get_even_over_9_mask = (~under_10_mask)&(even_mask)
    get_odd_over_9_mask = (~under_10_mask)&(~even_mask)
    
    dn[under_10_mask] = dn_list[n[under_10_mask]]
    dn[get_even_over_9_mask] = n[get_even_over_9_mask]/(n[get_even_over_9_mask]+3.8)
    dn[get_odd_over_9_mask] = n[get_odd_over_9_mask]/(n[get_odd_over_9_mask]+1.4)

    return dn
        
## this is as optimized as I can manage at the moment
## if this is really difficult, reconsider numba
def vectorized_qn_scale(x, c=2.219144465985076,stable_eps=10**-12):
    # takes a (N,k) array of observations and computes a Qn estimate for each set
    
    n = x.shape[1]
    if n < 2:
        return np.array([np.NaN for i in range(x.shape[0])])
    nan_count = np.sum(np.isnan(x),axis=1)
    n_effective = n-nan_count
    
    undersampled_mask = (n_effective<2)
    n_well_sampled = n_effective[~undersampled_mask]    
    x_well_sampled = x[~undersampled_mask]
    Qn_output = np.zeros((x.shape[0],))
    Qn_output[undersampled_mask] = np.NaN
    
    residual_nan_count = np.sum(np.isnan(x_well_sampled),axis=1)
    has_nans = (np.sum(residual_nan_count)>0)
    
    dn = dn_fn(n_well_sampled)
    
    vec_xi_minus_xj_mat = np.abs(x_well_sampled[:,:,None]-x_well_sampled[:,None,:])+stable_eps
    vec_xi_minus_xj_mat_diag = np.triu(vec_xi_minus_xj_mat,k=1)
    upper_right = vec_xi_minus_xj_mat_diag[vec_xi_minus_xj_mat_diag!=0]

    n_comparisons = ((x_well_sampled.shape[1]-1)*x_well_sampled.shape[1])//2 
    pair_arr = upper_right.reshape(vec_xi_minus_xj_mat.shape[0],n_comparisons)
    
    h = np.floor(n_effective/2).astype(int) + 1
    q = (h*(h-1))//2
    q = q-1 # to account for base 0
    q[q>pair_arr.shape[1]] = pair_arr.shape[1]-1
    
    if has_nans:
        k = np.array([np.partition(pair_arr[i],q[i])[q[i]] for i in range(x_well_sampled.shape[0])])
    else:
        q_single_val = q[0]
        k = np.partition(pair_arr,q_single_val,axis=1)[:,q_single_val] ## NaNs are considered inf
        
    Qn = dn*c*k
    Qn_output[~undersampled_mask] = Qn
    
    return Qn_output

## partner function for vectorized_qn_scale that uses the statsmodels implemenation
## of the Rousseeuw and Croux algorithm for finding Qn
## it has been checked for consistency with vectorized_qn_scale so it should
## produce the same results. Flip to using this when the sample size exceeds 200

def RandC_algo_qn_scale(x, c=2.219144465985076):
    n = x.shape[1]        
    nan_count = np.sum(np.isnan(x),axis=1)
    n_effective = n-nan_count
    
    undersampled_mask = (n_effective<2)
    n_well_sampled = n_effective[~undersampled_mask]    
    x_well_sampled = x[~undersampled_mask]
    Qn_output = np.zeros((x.shape[0],))
    Qn_output[undersampled_mask] = np.NaN
    
    residual_nan_count = np.sum(np.isnan(x_well_sampled),axis=1)
    has_nans = (np.sum(residual_nan_count)>0)
    
    dn = dn_fn(n_well_sampled)
    if has_nans:
        Qn = []
        for i in range(x_well_sampled.shape[0]):
            x_well_sampled_nonan = x_well_sampled[i][~np.isnan(x_well_sampled[i])]
            Qn.append(dn[i]*qn_scale(x_well_sampled_nonan,c=c,axis=0))
        Qn = np.array(Qn)
    else:
        Qn = dn*qn_scale(x_well_sampled,c=c,axis=1)
    
    Qn_output[~undersampled_mask] = Qn
    
    return Qn_output

def qn_scale_fastest_algo(x, c=2.219144465985076, switch_thr=200, axis=None):
    # takes a (N,k) array of observations and computes a Qn estimate for each set
    # picks the best algorithm for computing this by switching to the better scaling one
    # at a sample size of 200
    # axis included for consistency
    
    if x.shape[1] > switch_thr:
        Qn_output = RandC_algo_qn_scale(x,c=c)
    else:
        Qn_output = vectorized_qn_scale(x,c=c)
    
    return Qn_output

def internal_CV_Qn(x,axis=None):
    # takes a (N,k) array of observations and computes an internal Cv (Qn/mean) estimate for each set
    # axis included for consistency
    Qn = qn_scale_fastest_algo(x)
    mu = np.nanmean(x,axis=1)
    int_cv_qn = Qn/mu
    return int_cv_qn

def Qn_pearson_estimate(x,ddof=None):
    # takes (N,t,2) values and computes N Qn pearson r estimates
    # ddof included for comparability with pearson_bivariate_estimator

    Qn_estimate_x = np.stack([qn_scale_fastest_algo(x[:,:,i]) for i in range(x.shape[2])],axis=1)# N,2 estimates of the individual variable scale =
    x_std = x/Qn_estimate_x[:,None,:] # N,t,2 standardized values
    
    add_var_estimate = qn_scale_fastest_algo(x_std[:,:,0] + x_std[:,:,1])**2 # N additive estimates
    sub_var_estimate = qn_scale_fastest_algo(x_std[:,:,0] - x_std[:,:,1])**2 # N subtractive estimates
    r = (add_var_estimate-sub_var_estimate)/(add_var_estimate+sub_var_estimate)
    return r

def Qn_pearson_estimate_same_scale(x,ddof=None):
    # takes (N,t,2) values and computes N Qn pearson r estimates
    # ddof included for comparability with pearson_bivariate_estimator
    # same scale meaning that the variances of the input variables are
    # assumed to be the same (as in the case of an acf)    
    add_var_estimate = qn_scale_fastest_algo(x[:,:,0] + x[:,:,1])**2 # N estimates
    sub_var_estimate = qn_scale_fastest_algo(x[:,:,0] - x[:,:,1])**2 # N estimates
    r = (add_var_estimate-sub_var_estimate)/(add_var_estimate+sub_var_estimate)
    return r

def vec_cov_axis_1(x,ddof=1):
    #https://stackoverflow.com/questions/40394775/vectorizing-numpy-covariance-for-3d-array
    #computes (N,k,k) covariance matrices for array (N,t,k)
    #can handle NaNs
    x_copy = x.copy()
    N = x_copy.shape[1] #fixed
    nan_count = np.sum(np.any(np.isnan(x_copy),axis=2,keepdims=True),axis=1,keepdims=True)
    N_without_nan = N-nan_count
    x_copy[np.isnan(x_copy)] = 0.
    m1 = x - np.sum(x_copy,axis=1,keepdims=True)/N_without_nan
    m1[np.isnan(m1)] = 0.
    cov = np.einsum('ikj,ikl->ijl',m1,m1)  / (N_without_nan - ddof)
    return cov

def vec_pearson_r_axis_1(x,ddof=1):
    # takes a (N,t,k) array and computes an (N,k,k) pearson correlation matrix
    cov = vec_cov_axis_1(x,ddof=ddof)
    cov_diag = np.diagonal(cov,axis1=1,axis2=2)
    root_cov_diag = np.sqrt(cov_diag)
    pearson_matrix = cov/root_cov_diag[:,:,np.newaxis]
    pearson_matrix = pearson_matrix/root_cov_diag[:,np.newaxis,:]
    return pearson_matrix

def pearson_bivariate_estimator(x,ddof=1):
    # takes a (N,t,2) array and computes an (N,) bivariate pearson r array
    pearson_matrix = vec_pearson_r_axis_1(x,ddof=ddof)
    pearson_r = pearson_matrix[:,1,0]
    return pearson_r

def ccf_estimator_over_variants(x,max_ccf_lag=5,ddof=1,pearson_estimator=Qn_pearson_estimate,**estkwargs):
    # can be used for acfs
    # takes a list of k (N,t,2) arrays, with t not necessarily being the same for each set
    # Converts it into a set of lagged (N,sum_k(t-lag),2) arrays and computes the lagged ccfs
    # then concatenates the k (N,) outputs into n (N,k) output
    ccf_output = []
    for lag in range(0,max_ccf_lag+1):
        lag_arr = []
        for trench_arr in x:
            if trench_arr.shape[1] >= lag:
                trench_lag_arr = np.stack([trench_arr[:,:trench_arr.shape[1]-lag,0],trench_arr[:,lag:,1]],axis=2) # a lagged (N,t-lag,2) array
                lag_arr.append(trench_lag_arr)
        if len(lag_arr)>0:
            lag_arr = np.concatenate(lag_arr,axis=1)
            pearson_r_output = pearson_estimator(lag_arr,ddof=ddof,**estkwargs)
        else:
            pearson_r_output = np.repeat(np.NaN,x[0].shape[0])
        ccf_output.append(pearson_r_output)
    ccf_output = np.stack(ccf_output,axis=1)
    return ccf_output

def bootstrap_density(trench_df,estimator,param_groups,return_kde=False,n_bootstraps_trench_density=1000,**estkwargs):        
    params = list(set.union(*[set(param_group) for param_group in param_groups]))
    param_group_positions = [[params.index(param) for param in param_group] for param_group in param_groups]
    n_samples = len(trench_df)
    sample_list = list(range(n_samples))
    sample_array = np.random.choice(sample_list,size=(n_bootstraps_trench_density,n_samples))
    val_arr = trench_df[params].values
    val_sample = val_arr[sample_array]
    ## check if single parameter estimator (assume it can set the axis), else assume it has the same format as the pearson estimator
    if len(param_groups[0]) == 1:
        estimate_sample = np.stack([estimator(val_sample[:,:,param_group_position[0]],axis=1,**estkwargs) for param_group_position in param_group_positions],axis=1)
    else:
        estimate_sample = np.stack([estimator(val_sample[:,:,param_group_position],**estkwargs) for param_group_position in param_group_positions],axis=1)
    if return_kde:
        try:
            estimate_kde = [sp.stats.gaussian_kde(estimate_sample[:,i][~np.isnan(estimate_sample[:,i])]) for i in range(estimate_sample.shape[1])]
            return estimate_kde
        except:
            return None
    else:
        return estimate_sample

def tapered_block_bootstrap_density(trench_df,estimator,param_groups,return_kde=False,bootstrap_block_len=5,n_bootstraps_trench_density=1000,**estkwargs):
    ## currently estimators must accept NaN values
    ## param_groups must be a list of lists of grouped parameters for grouped statistics
    ## Can only take mother values
    
    params = list(set.union(*[set(param_group) for param_group in param_groups]))
    param_group_positions = [[params.index(param) for param in param_group] for param_group in param_groups]
    
    if np.sum(trench_df["Mother"]) == 0:
        return None
    
    mother_df = trench_df[trench_df["Mother"]]
    cell_cycle_timeseries = mother_df[params].values
    while bootstrap_block_len>0:
        if cell_cycle_timeseries.shape[0] > bootstrap_block_len:
            tapered_bootstrap_sample = tapered_block_bootstrap(cell_cycle_timeseries,bootstrap_block_len,n_bootstraps_trench_density)
            if len(params) == 1:
                tapered_bootstrap_sample = tapered_bootstrap_sample[:,:,np.newaxis]
                
            if len(param_groups[0]) == 1:
                estimate_sample = np.stack([estimator(tapered_bootstrap_sample[:,:,param_group_position[0]],axis=1,**estkwargs) for param_group_position in param_group_positions],axis=1)
                
            else:
                estimate_sample = np.stack([estimator(tapered_bootstrap_sample[:,:,param_group_position],**estkwargs) for param_group_position in param_group_positions],axis=1)
            
            if return_kde:            
                try:
                    estimate_kde = [sp.stats.gaussian_kde(estimate_sample[:,i][~np.isnan(estimate_sample[:,i])]) for i in range(estimate_sample.shape[1])]
                    return estimate_kde
                except:
                    return None
            else:
                return estimate_sample
        else:
            bootstrap_block_len -= 1
    return None

def get_trajectory_array(trench_df,trench_df_cellid_indexed):
    # get first and second generation cellids
    # trench_df_cellid_indexed = trench_df.set_index("Global CellID")
    mother_df = trench_df[trench_df["Mother"]]
    mother_cellids = mother_df["Global CellID"].tolist()
    daughter_2_gen_1_cellids = mother_df["Daughter CellID 2"].tolist()[:-1]
    all_cellids = trench_df["Global CellID"].tolist()
    all_cellids = [idx for idx in all_cellids if idx != -1]
    daughter_2_gen_1_cellids = [gen_1_cellid if gen_1_cellid in all_cellids else -1 for gen_1_cellid in daughter_2_gen_1_cellids]
    
    cellid_to_daughter_1_dict = trench_df_cellid_indexed["Daughter CellID 1"].to_dict()
    cellid_to_daughter_1_dict[-1] = -1
    
    daughter_2_gen_2_cellids = [cellid_to_daughter_1_dict[cellid] for cellid in daughter_2_gen_1_cellids[:-1]]
    daughter_2_gen_2_cellids = [gen_2_cellid if gen_2_cellid in all_cellids else -1 for gen_2_cellid in daughter_2_gen_2_cellids]
        
    # construct trajectory array with mother trajectory in the bottom row
    # nonexistant cellids take value -1
    trajectory_arr = np.repeat(np.array(mother_cellids)[np.newaxis,:],len(mother_cellids),axis=0)
    np.fill_diagonal(trajectory_arr[:-1,1:],daughter_2_gen_1_cellids)
    np.fill_diagonal(trajectory_arr[:-2,2:],daughter_2_gen_2_cellids)
    trajectory_arr[np.triu_indices(trajectory_arr.shape[0],k=3)] = -1
    trajectory_arr[:-1][np.tril_indices(trajectory_arr.shape[0]-1,k=-1)] = -1
    
    return trajectory_arr

## signature is a super useful argument! See following for usen_bootstraps_trench_density
## Make this faster by reindexing the cellids
def get_trajectory_timeseries(trench_df,params):
    ## lower integer values of global cellids
    trench_df_cellid_indexed = trench_df.set_index("Global CellID")
    trajectory_arr = get_trajectory_array(trench_df,trench_df_cellid_indexed)
    cellid_params_dict = dict(zip(trench_df_cellid_indexed.index.tolist(),trench_df_cellid_indexed[params].values))
    cellid_params_dict[-1] = np.array([np.NaN for param in params])
    cellid_params_vec = np.vectorize(cellid_params_dict.__getitem__,signature='()->(n)')
    trajectory_timeseries = cellid_params_vec(trajectory_arr)
    return trajectory_timeseries

def stationary_bootstrap_density(trench_df,estimator,param_groups,return_kde=False,bootstrap_block_len=5,n_bootstraps_trench_density=1000,**estkwargs):
    ## currently estimators must accept NaN values
    ## param_groups must be a list of lists of grouped parameters for grouped statistics
    ## can also return kdes for each trench if necessary for resampling
    ## the stationary bootstrap uses the data from both the mother cell and the first daughter lineage
    ## other parts of the lineage are omitted for simplicity since they are highly correlated with these two
    ## components anyways. The stationary bootstrap should NOT be used to estimate the ACF since
    ## the discontinuities of values at the block joints will create a bias in the ACF estimator
    ## For this purpose, I favor using the tapered block bootstrap, which has the limitation of 
    ## only using the mother cell data, but that should be fine for the ACF which only uses
    ## the mother cell anyways.
    
    params = list(set.union(*[set(param_group) for param_group in param_groups]))
    param_group_positions = [[params.index(param) for param in param_group] for param_group in param_groups]
    
    if np.sum(trench_df["Mother"]) == 0:
        return None
    
    trajectory_timeseries = get_trajectory_timeseries(trench_df,params)
    while bootstrap_block_len>0:
        if trajectory_timeseries.shape[1] > bootstrap_block_len:
            flat_trajectory_timeseries = trajectory_timeseries.swapaxes(0,1).reshape(trajectory_timeseries.shape[0],-1)
            stationary_bootstrap_sample = stationary_bootstrap(flat_trajectory_timeseries,bootstrap_block_len,n_bootstraps_trench_density)
            stationary_bootstrap_sample = stationary_bootstrap_sample.reshape(stationary_bootstrap_sample.shape[0],\
                                            stationary_bootstrap_sample.shape[1],trajectory_timeseries.shape[1],trajectory_timeseries.shape[2]).swapaxes(1,2)
            #note that the following steps eliminate the order of the samples, should not be used for ACFs!
            stationary_bootstrap_sample = stationary_bootstrap_sample.reshape(stationary_bootstrap_sample.shape[0],-1,stationary_bootstrap_sample.shape[3])
            stationary_bootstrap_sample_nancount = np.sum(np.isnan(stationary_bootstrap_sample[:,:,0]),axis=1)
            stationary_bootstrap_sample_minnan = np.min(stationary_bootstrap_sample_nancount)
            
            if stationary_bootstrap_sample_minnan > 0:
                stationary_bootstrap_sample_trim_argsort = np.argsort(stationary_bootstrap_sample[:,:,0],axis=1)
                stationary_bootstrap_sample_trimmed = np.take_along_axis(stationary_bootstrap_sample, stationary_bootstrap_sample_trim_argsort[:,:,None], axis=1)[:,:-stationary_bootstrap_sample_minnan]
            else:
                stationary_bootstrap_sample_trimmed = stationary_bootstrap_sample
                        
            if len(param_groups[0]) == 1:
                estimate_sample = np.stack([estimator(stationary_bootstrap_sample_trimmed[:,:,param_group_position[0]],axis=1,**estkwargs) for param_group_position in param_group_positions],axis=1)
            else:
                estimate_sample = np.stack([estimator(stationary_bootstrap_sample_trimmed[:,:,param_group_position],**estkwargs) for param_group_position in param_group_positions],axis=1)
            if return_kde:
                try:
                    estimate_kde = [sp.stats.gaussian_kde(estimate_sample[:,i][~np.isnan(estimate_sample[:,i])]) for i in range(estimate_sample.shape[1])]
                    return estimate_kde
                except:
                    return None
            else:
                return estimate_sample
        else:
            bootstrap_block_len -= 1
    return None

def tapered_block_bootstrap_ccf_density(mother_df,pearson_estimator,param_groups,return_kde=False,max_ccf_lag=3,bootstrap_block_len=6,n_bootstraps=1000,**estkwargs):
    ## A tapered block bootstrap defined across a variant df
    ## variant df must be sorted by timepoint for each trench, this can be done by sorting global cellids
    ## currently estimators must accept NaN values
    ## param_groups must be a list of lists of grouped parameters for grouped statistics
    ## Can only take mother values
    ## bootstrap blocks should be larger than acf lag (good rule of thumb is 4x the length, but 2x also fine)
    ## note that this bootstrap is over the entire variant set since the sampling per variant is poor
    ## still, I have added resampling of the trenches on top of this
    
    params = list(set.union(*[set(param_group) for param_group in param_groups]))

    cell_cycle_timeseries_df = mother_df[params]
    timepoint_counts = cell_cycle_timeseries_df.groupby("Multi-Experiment Phenotype Trenchid").size() #note that this counts NaNs
    
    ## filtering for examples at least twice the length of the chosen block size
    ## just picked this because it seems to control the variance outliers, perhaps a more principled choice is possible
    trenches_longer_than_bootstrap = timepoint_counts>(2*bootstrap_block_len)
    cell_cycle_timeseries_df_filtered = cell_cycle_timeseries_df.loc[trenches_longer_than_bootstrap[trenches_longer_than_bootstrap].index.tolist()]
    trench_idx_list = cell_cycle_timeseries_df_filtered.index.get_level_values(0).unique()
    n_trenches = len(trench_idx_list)
    
    tapered_bootstrap_sample_all_variants = []
    for i,trench_idx in enumerate(trench_idx_list):
        cell_cycle_timeseries = cell_cycle_timeseries_df_filtered.loc[trench_idx][params].values
        tapered_bootstrap_sample = tapered_block_bootstrap(cell_cycle_timeseries,bootstrap_block_len,n_bootstraps)
        if len(params) == 1:
            tapered_bootstrap_sample = tapered_bootstrap_sample[:,:,np.newaxis] #N, t, #param bootstrap
        tapered_bootstrap_sample = xr.DataArray(tapered_bootstrap_sample,dims=["Bootstrap","Timepoint","Parameter"],\
                                            coords=[range(n_bootstraps),range(tapered_bootstrap_sample.shape[1]),params])
        tapered_bootstrap_sample = tapered_bootstrap_sample.to_dataframe(name="Value").reset_index()
        tapered_bootstrap_sample["Bootstrap"] = tapered_bootstrap_sample["Bootstrap"]+(i*n_bootstraps)
        tapered_bootstrap_sample_all_variants.append(tapered_bootstrap_sample)
    
    if len(tapered_bootstrap_sample_all_variants) == 0:
        return None
    
    tapered_bootstrap_sample_all_variants = pd.concat(tapered_bootstrap_sample_all_variants).set_index(["Bootstrap","Timepoint","Parameter"])
    tapered_bootstrap_sample_all_variants = tapered_bootstrap_sample_all_variants.to_xarray().to_array()[0]
    
    ccf_estimate_sample = []
    for param_group in param_groups:
        working_sample = tapered_bootstrap_sample_all_variants.loc[:,:,param_group].to_numpy()
        bootstrap_roll = np.random.choice(np.arange(0,working_sample.shape[0]),size=(n_bootstraps*n_trenches))
        bootstrap_roll = working_sample[bootstrap_roll]
        del working_sample
        bootstrap_roll = bootstrap_roll.reshape(n_trenches,n_bootstraps,bootstrap_roll.shape[1],2)
        bootstrap_roll = [bootstrap_roll[k] for k in range(bootstrap_roll.shape[0])]
        
        ccf_output = ccf_estimator_over_variants(bootstrap_roll,max_ccf_lag=max_ccf_lag,pearson_estimator=pearson_estimator,**estkwargs)
        ccf_estimate_sample.append(ccf_output)
    ccf_estimate_sample = np.stack(ccf_estimate_sample,axis=1)

    if return_kde:            
        try:
            estimate_kde = []
            for i in range(ccf_estimate_sample.shape[1]):
                estimate_kde_lag_list = []
                for j in range(1,ccf_estimate_sample.shape[2]):
                    estimate_kde_lag = sp.stats.gaussian_kde(ccf_estimate_sample[:,i,j][~np.isnan(ccf_estimate_sample[:,i,j])]) 
                    estimate_kde_lag_list.append(estimate_kde_lag)
                estimate_kde.append(estimate_kde_lag_list)
            return estimate_kde
        except:
            return None
    
    else:
        return ccf_estimate_sample

def all_trench_bootstrap(trench_df,bootstrap_density_function,param_groups_list,return_kde=False,estimators=[np.nanmean],estimator_names=["Mean"],\
                         n_bootstraps_trench_density=1000,variant_index="oDEPool7_id",trench_index="Multi-Experiment Phenotype Trenchid",\
                         cell_index="Multi-Experiment Global CellID",**kwargs):
    
    estimate_samples_df = []
    for i,estimator in enumerate(estimators):
        param_groups = param_groups_list[i]
        param_groups_joined = ["-".join(param_group) for param_group in param_groups]
        
        #sort again since later sgRNA sort messes up order
        trench_df = trench_df.reset_index().set_index(cell_index).sort_index().reset_index().set_index(trench_index)
        estimate_sample = bootstrap_density_function(trench_df,estimator,param_groups,return_kde=return_kde,n_bootstraps_trench_density=n_bootstraps_trench_density,**kwargs) #N x K output
        
        if estimate_sample is None:
            pass
        else:
            if return_kde:
                estimate_sample_df = xr.DataArray(estimate_sample,coords=[param_groups_joined],\
                                                  dims=["Variable(s)"],name="KDE").to_dataframe()
            else:
                estimate_sample_df = xr.DataArray(estimate_sample,coords=[range(0,n_bootstraps_trench_density),param_groups_joined],\
                                                  dims=["Trench Bootstrap Sample","Variable(s)"],name="Value").to_dataframe()
            estimate_sample_df["Estimator"] = estimator_names[i]
            estimate_samples_df.append(estimate_sample_df)
            
    if len(estimate_samples_df)>0:
        estimate_samples_df = pd.concat(estimate_samples_df).reset_index()
        estimate_samples_df.index.name="Bootstrap Index"
        if len(trench_df) > 0:
            estimate_samples_df[variant_index] = trench_df.iloc[0][variant_index]
        else:
            estimate_samples_df[variant_index] = -1
        return estimate_samples_df
    else:
        return None

def get_trench_sample_df(trench_df,estimator_name,n_bootstraps_trench_density,n_bootstraps_trench_aggregate,param_groups_joined):
    
    sampled_trench_df = trench_df.reset_index().set_index(["Trench Bootstrap Sample","Estimator","Variable(s)"]).sort_index().reindex(labels=param_groups_joined,level="Variable(s)")
    sampled_trench_df = sampled_trench_df.reset_index().set_index("Trench Bootstrap Sample").loc[np.random.choice(range(n_bootstraps_trench_density),size=(n_bootstraps_trench_aggregate,))].reset_index()
    sampled_trench_df["Aggregate Trench Bootstrap Sample"] = np.repeat(range(n_bootstraps_trench_aggregate),len(param_groups_joined)*len(estimator_name_list))
    sampled_trench_df = sampled_trench_df.set_index(["Aggregate Trench Bootstrap Sample","Estimator","Variable(s)"])
    sampled_trench_df = sampled_trench_df.drop(columns=["Trench Bootstrap Sample","Bootstrap Index","oDEPool7_id","Multi-Experiment Phenotype Trenchid"])
    
    return sampled_trench_df

def EVE_intrinsic_variance_agg(Qn_arr,axis=0):
    ## x is (t,N) where t is trench, N is bootstraps
    intrinsic_variance = np.nanmedian(Qn_arr**2,axis=axis)
    return intrinsic_variance

def EVE_extrinsic_variance_agg(mean_arr,axis=0):
    ## x is (t,N) where t is trench, N is bootstraps
    extrinsic_variance = np.nanvar(mean_arr,axis=axis)
    return extrinsic_variance

def EVE_total_variance_agg(mean_Qn_arr,axis=0):
    mean_arr,Qn_arr = mean_Qn_arr[:,:,0],mean_Qn_arr[:,:,1]
    intrinsic_variance = np.nanmedian(Qn_arr**2,axis=axis)
    extrinsic_variance = np.nanvar(mean_arr,axis=axis)
    ttl_variance = intrinsic_variance+extrinsic_variance
    ttl_variance = ttl_variance[:,None]
    return ttl_variance

def Frac_EVE_ext_over_int_agg(mean_Qn_arr,axis=0):
    mean_arr,Qn_arr = mean_Qn_arr[:,:,0],mean_Qn_arr[:,:,1]
    intrinsic_variance = np.nanmedian(Qn_arr**2,axis=axis)
    extrinsic_variance = np.nanvar(mean_arr,axis=axis)
    frac_extrinsic = extrinsic_variance/intrinsic_variance
    frac_extrinsic = frac_extrinsic[:,None]
    return frac_extrinsic

def CV_total_agg(mean_Qn_arr,axis=0):
    mean_arr,Qn_arr = mean_Qn_arr[:,:,0],mean_Qn_arr[:,:,1]
    intrinsic_variance = np.nanmedian(Qn_arr**2,axis=axis)
    extrinsic_variance = np.nanvar(mean_arr,axis=axis)
    ttl_variance = intrinsic_variance+extrinsic_variance
    expected_value = np.nanmedian(mean_arr,axis=axis)
    CV_ttl = np.sqrt(ttl_variance)/expected_value
    CV_ttl = CV_ttl[:,None]
    return CV_ttl

def trench_aggregate_bootstrap(variant_df,param_groups_list,estimator_names=[["Mean"],["Qn"],["Mean"],["Mean","Qn"],["Mean","Qn"],["Mean","Qn"],\
                                                                            ["CV (Internal)"],["Pearson R"],["Qn Pearson R"]],\
                               aggregators=[np.nanmedian,EVE_intrinsic_variance_agg,EVE_extrinsic_variance_agg,EVE_total_variance_agg,Frac_EVE_ext_over_int_agg,\
                                            CV_total_agg,np.nanmedian,np.nanmedian,np.nanmedian],\
                               agg_names=["Mean","Variance (Intrinsic)","Variance (Extrinsic)","Variance (Total)","Fraction of Extrinsic Variance",\
                                          "CV (Total)","CV (Internal)","Pearson R","Qn Pearson R"],\
                             n_bootstraps_trench_aggregate=1000,trench_index="Multi-Experiment Phenotype Trenchid"):
    
    n_bootstraps_trench_density = variant_df["Trench Bootstrap Sample"].max()+1
    variant_df = variant_df.reset_index().set_index(["Estimator",trench_index]).sort_index()
        
    aggregate_samples_df = []
    for i,estimator_name_list in enumerate(estimator_names):
        param_groups = param_groups_list[i]
        param_groups_joined = ["-".join(param_group) for param_group in param_groups]
        
        #sort again since later sgRNA sort messes up order
        variant_estimator_df = variant_df.loc[estimator_name_list]
        trench_index_groupby = variant_estimator_df.groupby(trench_index)
        sampled_variant_df = trench_index_groupby.apply(lambda x: get_trench_sample_df(x,estimator_name_list,n_bootstraps_trench_density,n_bootstraps_trench_aggregate,param_groups_joined))        
        aggregate_sample = aggregators[i](sampled_variant_df.to_xarray().reindex(indexers={"Variable(s)":param_groups_joined,\
                                                                "Estimator":estimator_name_list}).to_array().values[0],axis=0)[:,0]
        aggregate_sample_df = xr.DataArray(aggregate_sample,coords=[range(0,n_bootstraps_trench_aggregate),param_groups_joined],dims=["Bootstrap Sample","Variable(s)"],name="Value").to_dataframe()
        aggregate_sample_df["Estimator"] = agg_names[i]
        aggregate_samples_df.append(aggregate_sample_df)
        
    aggregate_samples_df = pd.concat(aggregate_samples_df).reset_index()
    aggregate_samples_df.index.name="Bootstrap Index"
    return aggregate_samples_df

def all_variant_ccf_bootstrap(variant_df,bootstrap_density_function,param_groups_list,pearson_estimators=[pearson_bivariate_estimator],estimator_names=["Pearson R"],\
                         n_bootstraps=1000,max_ccf_lag=3,bootstrap_block_len=6,variant_index="oDEPool7_id",trench_index="Multi-Experiment Phenotype Trenchid",\
                              timepoint_index="initial timepoints",**kwargs):
    
    #sort again since later sgRNA sort messes up order
    mother_df = variant_df[variant_df["Mother"]]
    mother_df = mother_df.reset_index().set_index([trench_index,timepoint_index]).sort_index()
        
    estimate_samples_df = []
    for i,pearson_estimator in enumerate(pearson_estimators):
        param_groups = param_groups_list[i]
        param_groups_joined = ["-".join(param_group) for param_group in param_groups]
        
        estimate_sample = bootstrap_density_function(mother_df,pearson_estimator,param_groups,n_bootstraps=n_bootstraps,max_ccf_lag=max_ccf_lag,\
                                                     bootstrap_block_len=bootstrap_block_len,**kwargs) #N x K output
        estimate_sample_df = xr.DataArray(estimate_sample,coords=[range(0,n_bootstraps),param_groups_joined,range(0,max_ccf_lag+1)],\
                                          dims=["Variant Bootstrap Sample","Variable(s)","Lag"],name="Value").to_dataframe()
        estimate_sample_df["Estimator"] = estimator_names[i]
        estimate_samples_df.append(estimate_sample_df)

    estimate_samples_df = pd.concat(estimate_samples_df).reset_index()
    estimate_samples_df.index.name="Bootstrap Index"
    estimate_samples_df[variant_index] = variant_df.index[0]
    
    estimate_samples_df = estimate_samples_df.drop(columns=[variant_index])
    
    return estimate_samples_df

##note that this may be broken for the control bootstrapping, have not tested this with the .sort_index call on the second line
def get_extrinsic_CV(partition,control_medians,value_col_name="Value"):
    input_columns = partition.columns
    partition = partition.set_index("Variable(s)").sort_index()
    index_list = partition.index.unique().tolist()
    partition[value_col_name] = np.sqrt(partition[value_col_name])/control_medians[index_list]
    partition["Estimator"] = "CV (Extrinsic)"
    partition = partition.reset_index()
    partition = partition.reindex(columns=input_columns)
    return partition

def cull_empty_partitions(df):
    ll = list(df.map_partitions(len).compute())
    df_delayed = df.to_delayed()
    df_delayed_new = list()
    pempty = None
    for ix, n in enumerate(ll):
        if 0 == n:
            pempty = df.get_partition(ix)
        else:
            df_delayed_new.append(df_delayed[ix])
    if pempty is not None:
        df = dd.from_delayed(df_delayed_new, meta=pempty)
    return df

## 1) Computing Estimators
- Actually computes estimators for different steady state values
- Will be merged with p-values, CIs and variances from bootstrapping

In [ ]:
dask_controller = tr.trcluster.dask_controller(
    walltime="03:00:00",
    local=False,
    n_workers=100,
    n_workers_min=100,
    memory="8GB",
    working_directory="/home/de64/scratch/de64/dask",
)
dask_controller.startdask()

In [ ]:
dask_controller.daskclient

### Reload Dataframes

In [ ]:
cell_cycle_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Cell_Cycle_df/", engine="pyarrow",calculate_divisions=True)
cell_cycle_df_preinduction = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Cell_Cycle_df/", engine="pyarrow",calculate_divisions=True)

timepoints_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Timepoints_df/", engine="pyarrow",calculate_divisions=True)
timepoints_df_preinduction = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Timepoints_df/", engine="pyarrow",calculate_divisions=True)

growth_df_ss = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Growth_df/", engine="pyarrow",calculate_divisions=True)
growth_df_preinduction = dd.read_parquet("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Growth_df/", engine="pyarrow",calculate_divisions=True)

### 1a) Calculate Location, Scale and Correlation Estimates

In [ ]:
def get_estimator_df_single_trench(trench_df,param_groups,estimator,final_columns,estimator_name,trench_index="Multi-Experiment Phenotype Trenchid"):
    param_groups_joined = ["-".join(param_group) for param_group in param_groups]
    estimator_output_list = []
    for param_group in param_groups:
        if len(param_group) == 1:
            estimator_output = estimator(trench_df[param_group].values.T, axis=1)[0]
        else:
            estimator_output = estimator(trench_df[param_group].values[None,:,:])[0]
        estimator_output_list.append(estimator_output)
    estimator_output = pd.DataFrame([estimator_output_list]).rename(columns={i:param_groups_joined[i] for i in range(len(estimator_output_list))})
    estimator_output[trench_index] = trench_df.index[0]
    estimator_output = estimator_output.set_index(trench_index)
    estimator_output[final_columns] = trench_df.iloc[0][final_columns]
    estimator_output["Estimator"] = estimator_name
    return estimator_output

def get_estimator_df_all_trenches(df,estimator_names,estimators,param_groups_list,final_columns,trench_index="Multi-Experiment Phenotype Trenchid"):
    trench_groupby = df.groupby(trench_index)
    trench_estimator_df_output = []
    for i,estimator_name in enumerate(estimator_names):
        estimator = estimators[i]
        param_groups = param_groups_list[i]
        
        trench_estimator_df = trench_groupby.apply(lambda x: get_estimator_df_single_trench(x,param_groups,estimator,\
                                                final_columns,estimator_name,trench_index=trench_index)).compute()
        
        trench_estimator_df_output.append(trench_estimator_df)
    # trench_estimator_df_output = pd.concat(trench_estimator_df_output).droplevel(1).reset_index().set_index(["Estimator",trench_index]) # DANIEL'S VERSION
    trench_estimator_df_output = pd.concat(trench_estimator_df_output).reset_index().set_index(["Estimator",trench_index]) # LUIS' VERSION
    return trench_estimator_df_output

def get_estimator_df_single_variant(variant_df,param_groups,estimator_name,final_columns,aggregator,is_paired_aggregetor,agg_name,trench_index="Multi-Experiment Phenotype Trenchid"):
    variant_df = variant_df.reset_index().set_index(["Induction","Estimator",trench_index]).sort_index()
    post_in_index = ("Post",estimator_name) in variant_df.index
    pre_in_index = ("Pre",estimator_name) in variant_df.index
    
    if post_in_index:
        if not pre_in_index and is_paired_aggregetor:
            return None
        
        post_estimator_variant_df = variant_df.loc["Post",estimator_name]
        
        if pre_in_index:
            pre_estimator_variant_df = variant_df.loc["Pre",estimator_name]
        param_groups_joined = ["-".join(param_group) for param_group in param_groups]

        estimator_series_output = {}
        for i,param_group in enumerate(param_groups_joined):
            post_param_arr = np.array(post_estimator_variant_df[param_group].tolist())[:,None,None]
            if is_paired_aggregetor:
                pre_param_arr = np.array(pre_estimator_variant_df[param_group].tolist())[:,None,None]
                estimator_output = aggregator(pre_param_arr,post_param_arr,axis=0)[0,0]
            else:
                estimator_output = aggregator(post_param_arr,axis=0)[0,0]
            estimator_series_output[param_group] = estimator_output
        estimator_output = pd.Series(estimator_series_output)
        del estimator_series_output

        estimator_output = pd.DataFrame(estimator_output).reset_index().rename(columns={"index":"Variable(s)",0:"Value"})
        estimator_output["Estimator"] = agg_name
        estimator_output[final_columns] = variant_df.iloc[0][final_columns]
    
        return estimator_output
    
    else:
        return None

def get_estimator_df(df,final_columns,param_groups_to_agg_list,estimator_names_to_agg,aggregators,is_paired_aggregetor_list,agg_names,\
                     sgrna_key='oDEPool7_id',trench_key='Multi-Experiment Phenotype Trenchid'):
    
    ## filter for trenches present pre and post treatment
    pre_post_mask = df.groupby(trench_key).apply(lambda x: sorted(x["Induction"].unique())==["Post","Pre"]).compute()
    trenchids_prepost = pre_post_mask[pre_post_mask].index.tolist()
    pre_post_df = df.loc[trenchids_prepost]
    
    sgrna_sorted = df.reset_index().set_index(sgrna_key).persist()
    sgrna_sorted_pre_post = pre_post_df.reset_index().set_index(sgrna_key).persist()
    
    sgrna_groupby = sgrna_sorted.groupby(sgrna_key)
    sgrna_groupby_pre_post = sgrna_sorted_pre_post.groupby(sgrna_key)
    
    first_idx = sgrna_sorted.index.unique().compute()[0]
    test_input = sgrna_sorted.loc[first_idx].compute()
    
    first_idx_pre_post = sgrna_sorted_pre_post.index.unique().compute()[0]
    test_input_pre_post = sgrna_sorted_pre_post.loc[first_idx_pre_post].compute()
    
    estimator_df_output = []
    for i,agg_name in enumerate(agg_names):
        param_groups = param_groups_to_agg_list[i]
        estimator_name = estimator_names_to_agg[i]
        aggregator = aggregators[i]
        is_paired_aggregetor = is_paired_aggregetor_list[i]
        
        if is_paired_aggregetor:
            working_test_input = test_input_pre_post
            working_sgrna_groupby = sgrna_groupby_pre_post
        else:
            working_test_input = test_input
            working_sgrna_groupby = sgrna_groupby
        
        test_output = get_estimator_df_single_variant(test_input,param_groups,estimator_name,\
                        final_columns,aggregator,is_paired_aggregetor,agg_name)
        estimator_df = working_sgrna_groupby.apply(lambda x: get_estimator_df_single_variant(x,param_groups,estimator_name,\
                                                final_columns,aggregator,is_paired_aggregetor,agg_name),meta=test_output).compute()
        
        estimator_df_output.append(estimator_df)
    # estimator_df_output = pd.concat(estimator_df_output).drop(columns=sgrna_key).reset_index().drop(columns=["level_1"]).set_index(["Estimator","Variable(s)",sgrna_key]).sort_index() # DANIEL'S VERSION
    estimator_df_output = pd.concat(estimator_df_output).reset_index(drop=True).set_index(["Estimator","Variable(s)",sgrna_key]).sort_index() #LUIS' VERSION
    return estimator_df_output

def median_difference(pre_array,post_array,axis=0):
    return np.nanmedian(post_array-pre_array,axis=axis)
def mean_difference(pre_array,post_array,axis=0):
    return np.nanmean(post_array-pre_array,axis=axis)
def median_difference_variance(pre_array,post_array,axis=0):
    return np.nanmedian((post_array**2)-(pre_array**2),axis=axis)
def morgan_pitman_statistic(pre_array,post_array,spearman=False,axis=0):
    #spearman takes too long, but precomputing the rank before bootstrap could be a fix
    #moving on without implementing since this is already taking too long and pearson r
    #is more effecient anyways
    U = post_array+pre_array
    V = post_array-pre_array
    U_flat,V_flat = U.reshape(U.shape[0],-1),V.reshape(V.shape[0],-1)
    UV_flat = np.stack([U_flat,V_flat],axis=2)
    UV_flat = np.swapaxes(UV_flat,0,1)
    if spearman:
        UV_flat = sp.stats.rankdata(UV_flat, method='average', axis=1)
    r = pearson_bivariate_estimator(UV_flat)
    r = r.reshape(U.shape[1],U.shape[2])
    return r

In [ ]:
final_columns = ['oDEPool7_id', 'sgRNA','EcoWG1_id', 'Gene', 'N Mismatch', 'Category', 'TargetID']
final_columns_cell_cycle = final_columns + ['N Observations','N Observations Proliferating']
estimators = [np.nanmean,qn_scale_fastest_algo,internal_CV_Qn,pearson_bivariate_estimator,Qn_pearson_estimate]
estimator_names = ["Mean","Qn","CV (Intrinsic)","Pearson R","Qn Pearson R"]

estimator_names_to_agg = ["Mean","Mean","Qn","Mean","CV (Intrinsic)","Pearson R","Qn Pearson R","Mean","Mean","Qn","Mean","CV (Intrinsic)"]
unpaired_aggregators=[np.nanmedian,np.nanmean,EVE_intrinsic_variance_agg,EVE_extrinsic_variance_agg,np.nanmedian,np.nanmedian,np.nanmedian]
paired_aggregators=[median_difference,mean_difference,median_difference_variance,morgan_pitman_statistic,median_difference]
aggregators = unpaired_aggregators + paired_aggregators
is_paired_aggregetor_list=[False for i in range(len(unpaired_aggregators))]+[True for i in range(len(paired_aggregators))]
agg_names=["Mean (Robust)","Mean (True)","Variance (Intrinsic)","Variance (Extrinsic)",\
           "CV (Intrinsic)","Pearson R","Qn Pearson R","Mean Difference (Robust)","Mean Difference (True)",\
          "Variance Difference (Intrinsic)","Variance MorganPitman Statistic (Extrinsic)","CV Difference (Intrinsic)"]

#### Cell Cycle Estimates

In [ ]:
single_variable_list = ['Birth: major_axis_length', 'Division: major_axis_length',
       'Delta: major_axis_length','Septum Position Length Normalized','Birth: Volume',\
        'Division: Volume','Delta: Volume','Delta time (s)',\
                       'Birth: Number of Nucleoids', 'Division: Number of Nucleoids', \
                        'Birth: Total Nucleoid Area', 'Division: Total Nucleoid Area' ]

pearsonr_variable_list = [['Birth: major_axis_length','Delta: major_axis_length']]
wrapped_single_variable_list = [[variable] for variable in single_variable_list]
param_groups_list=[wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                   pearsonr_variable_list,pearsonr_variable_list]
param_groups_to_agg_list=[wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                   wrapped_single_variable_list,wrapped_single_variable_list,pearsonr_variable_list,pearsonr_variable_list,\
                    wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                    wrapped_single_variable_list]

steady_state_df_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Cell_Cycle_df"
preinduction_df_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Cell_Cycle_df"
trench_estimator_path = steady_state_df_path + "_Trench_Estimators.pkl"
estimator_path = steady_state_df_path + "_Estimators.pkl"

final_cell_cycle_df_ss = dd.read_parquet(steady_state_df_path, engine="pyarrow",calculate_divisions=True)
final_cell_cycle_df_ss = final_cell_cycle_df_ss[final_cell_cycle_df_ss["Proliferating"]]

final_cell_cycle_df_preinduction = dd.read_parquet(preinduction_df_path, engine="pyarrow",calculate_divisions=True)
final_cell_cycle_df_preinduction = final_cell_cycle_df_preinduction[final_cell_cycle_df_preinduction["Proliferating"]]

trench_estimator_ss_df_output = get_estimator_df_all_trenches(final_cell_cycle_df_ss,estimator_names,estimators,param_groups_list,final_columns_cell_cycle,\
                                                              trench_index="Multi-Experiment Phenotype Trenchid")
trench_estimator_preinduction_df_output = get_estimator_df_all_trenches(final_cell_cycle_df_preinduction,estimator_names,estimators,param_groups_list,\
                                                                        final_columns_cell_cycle,trench_index="Multi-Experiment Phenotype Trenchid")
trench_estimator_ss_df_output["Induction"] = "Post"
trench_estimator_preinduction_df_output["Induction"] = "Pre"
trench_estimator_ss_df_output = trench_estimator_ss_df_output.reset_index().set_index(["Induction","Estimator","Multi-Experiment Phenotype Trenchid"])
trench_estimator_preinduction_df_output = trench_estimator_preinduction_df_output.reset_index().set_index(["Induction","Estimator","Multi-Experiment Phenotype Trenchid"])

trench_estimator_df = pd.concat([trench_estimator_ss_df_output,trench_estimator_preinduction_df_output])
del trench_estimator_ss_df_output
del trench_estimator_preinduction_df_output
trench_estimator_df.to_pickle(trench_estimator_path)
del trench_estimator_df
trench_estimator_df = pd.read_pickle(trench_estimator_path)
trench_estimator_df = trench_estimator_df.reset_index()
trench_estimator_dd = dd.from_pandas(trench_estimator_df,100).persist();
trench_estimator_dd = trench_estimator_dd.set_index("Multi-Experiment Phenotype Trenchid").persist()
wait(trench_estimator_dd);
del trench_estimator_df

estimator_df = get_estimator_df(trench_estimator_dd,final_columns_cell_cycle,param_groups_to_agg_list,estimator_names_to_agg,aggregators,is_paired_aggregetor_list,agg_names,sgrna_key='oDEPool7_id')

###Adding derived values
### note that CV (external) is normalized by wt median

'''
LUIS: REMOVED BECAUSE THERE ARE NO CONTROLS
'''
'''
control_estimator_df = estimator_df[estimator_df["Category"].isin(['OnlyPlasmid', 'NoTarget'])]
control_medians = control_estimator_df.loc["Mean (Robust)"].groupby("Variable(s)")["Value"].median()

CV_extrinsic_df = estimator_df.loc["Variance (Extrinsic)"].copy()
CV_extrinsic_df["Value"] = np.sqrt(estimator_df.loc["Variance (Extrinsic)"]["Value"])/control_medians
CV_extrinsic_df["Estimator"] = "CV (Extrinsic)"
CV_extrinsic_df = CV_extrinsic_df.reset_index().set_index(["Estimator","Variable(s)","oDEPool7_id"])
'''
'''
'''

#estimator_df_output = pd.concat([estimator_df,CV_extrinsic_df]).sort_index() DANIEL'S VERSION
estimator_df_output = estimator_df.sort_index()
estimator_df_output.to_pickle(estimator_path)

In [ ]:
### HERE ENDS MAIN CODE

In [ ]:
# df = final_cell_cycle_df_ss
# estimator_names = estimator_names
# estimators = estimators
# param_groups_list = param_groups_list
# final_columns = final_columns_cell_cycle
# trench_index = "Multi-Experiment Phenotype Trenchid"

# trench_groupby = df.groupby(trench_index)
# trench_estimator_df_output = []
# for i,estimator_name in enumerate(estimator_names[:2]):
#     estimator = estimators[i]
#     param_groups = param_groups_list[i]

#     # print("Starting", i, estimator_name)

#     trench_estimator_df = trench_groupby.apply(lambda x: get_estimator_df_single_trench(x,param_groups,estimator,\
#                                             final_columns,estimator_name,trench_index=trench_index)).compute()

#     trench_estimator_df_output.append(trench_estimator_df)
#     # print("Done", i, estimator_name)
# trench_estimator_df_output = pd.concat(trench_estimator_df_output)#.droplevel(1).reset_index().set_index(["Estimator",trench_index]) #REAL
# # trench_estimator_df_output = pd.concat(trench_estimator_df_output).reset_index().set_index(["Estimator",trench_index])


In [ ]:
## get_estimator_df function:

# df = trench_estimator_dd
# final_columns= final_columns_cell_cycle
# param_groups_to_agg_list=param_groups_to_agg_list
# estimator_names_to_agg=estimator_names_to_agg
# aggregators=aggregators
# is_paired_aggregetor_list=is_paired_aggregetor_list
# agg_names=agg_names
# sgrna_key='oDEPool7_id'
# trench_key='Multi-Experiment Phenotype Trenchid'
    
# ## filter for trenches present pre and post treatment
# pre_post_mask = df.groupby(trench_key).apply(lambda x: sorted(x["Induction"].unique())==["Post","Pre"]).compute()
# trenchids_prepost = pre_post_mask[pre_post_mask].index.tolist()
# pre_post_df = df.loc[trenchids_prepost]

# sgrna_sorted = df.reset_index().set_index(sgrna_key).persist()
# sgrna_sorted_pre_post = pre_post_df.reset_index().set_index(sgrna_key).persist()

# sgrna_groupby = sgrna_sorted.groupby(sgrna_key)
# sgrna_groupby_pre_post = sgrna_sorted_pre_post.groupby(sgrna_key)

# first_idx = sgrna_sorted.index.unique().compute()[0]
# test_input = sgrna_sorted.loc[first_idx].compute()

# first_idx_pre_post = sgrna_sorted_pre_post.index.unique().compute()[0]
# test_input_pre_post = sgrna_sorted_pre_post.loc[first_idx_pre_post].compute()

# estimator_df_output = []
# for i,agg_name in enumerate(agg_names):
#     param_groups = param_groups_to_agg_list[i]
#     estimator_name = estimator_names_to_agg[i]
#     aggregator = aggregators[i]
#     is_paired_aggregetor = is_paired_aggregetor_list[i]

#     if is_paired_aggregetor:
#         working_test_input = test_input_pre_post
#         working_sgrna_groupby = sgrna_groupby_pre_post
#     else:
#         working_test_input = test_input
#         working_sgrna_groupby = sgrna_groupby

#     test_output = get_estimator_df_single_variant(test_input,param_groups,estimator_name,\
#                     final_columns,aggregator,is_paired_aggregetor,agg_name)
#     estimator_df = working_sgrna_groupby.apply(lambda x: get_estimator_df_single_variant(x,param_groups,estimator_name,\
#                                             final_columns,aggregator,is_paired_aggregetor,agg_name),meta=test_output).compute()

#     estimator_df_output.append(estimator_df)

# # estimator_df_output = pd.concat(estimator_df_output).drop(columns=sgrna_key).reset_index().drop(columns=["level_1"]).set_index(["Estimator","Variable(s)",sgrna_key]).sort_index() # DANIEL'S 
# # estimator_df_output = pd.concat(estimator_df_output).drop(columns=sgrna_key).reset_index().set_index(["Estimator","Variable(s)",sgrna_key]).sort_index() # LUIS'
# estimator_df_output = pd.concat(estimator_df_output).reset_index(drop=True).set_index(["Estimator","Variable(s)",sgrna_key]).sort_index()

#### Timepoint Estimates

In [ ]:
estimators = [np.nanmean,qn_scale_fastest_algo,internal_CV_Qn]
estimator_names = ["Mean","Qn","CV (Intrinsic)"]

estimator_names_to_agg = ["Mean","Mean","Qn","Mean","CV (Intrinsic)","Mean","Mean","Qn","Mean","CV (Intrinsic)"]
unpaired_aggregators = [np.nanmedian,np.nanmean,EVE_intrinsic_variance_agg,EVE_extrinsic_variance_agg,np.nanmedian]
paired_aggregators = [median_difference,mean_difference,median_difference_variance,morgan_pitman_statistic,median_difference]
aggregators = unpaired_aggregators + paired_aggregators
is_paired_aggregetor_list = [False for i in range(len(unpaired_aggregators))]+[True for i in range(len(paired_aggregators))]
agg_names = ["Mean (Robust)","Mean (True)","Variance (Intrinsic)","Variance (Extrinsic)",\
           "CV (Intrinsic)","Mean Difference (Robust)","Mean Difference (True)",\
          "Variance Difference (Intrinsic)","Variance MorganPitman Statistic (Extrinsic)","CV Difference (Intrinsic)"]

single_variable_list = ['major_axis_length', 'minor_axis_length','YFP mean_intensity', 'Number of Nucleoids', 'Total Nucleoid Area'] # ADD HERE?

wrapped_single_variable_list = [[variable] for variable in single_variable_list]
param_groups_list=[wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list]
param_groups_to_agg_list=[wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                    wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                    wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                    wrapped_single_variable_list]

steady_state_df_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Timepoints_df"
preinduction_df_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Timepoints_df"
trench_estimator_path = steady_state_df_path + "_Trench_Estimators.pkl"
estimator_path = steady_state_df_path + "_Estimators.pkl"

final_timepoints_df_ss = dd.read_parquet(steady_state_df_path, engine="pyarrow",calculate_divisions=True)
final_timepoints_df_preinduction = dd.read_parquet(preinduction_df_path, engine="pyarrow",calculate_divisions=True)

trench_estimator_ss_df_output = get_estimator_df_all_trenches(final_timepoints_df_ss,estimator_names,estimators,param_groups_list,final_columns,\
                                                              trench_index="Multi-Experiment Phenotype Trenchid")
trench_estimator_preinduction_df_output = get_estimator_df_all_trenches(final_timepoints_df_preinduction,estimator_names,estimators,param_groups_list,\
                                                                        final_columns,trench_index="Multi-Experiment Phenotype Trenchid")
trench_estimator_ss_df_output["Induction"] = "Post"
trench_estimator_preinduction_df_output["Induction"] = "Pre"
trench_estimator_ss_df_output = trench_estimator_ss_df_output.reset_index().set_index(["Induction","Estimator","Multi-Experiment Phenotype Trenchid"])
trench_estimator_preinduction_df_output = trench_estimator_preinduction_df_output.reset_index().set_index(["Induction","Estimator","Multi-Experiment Phenotype Trenchid"])

trench_estimator_df = pd.concat([trench_estimator_ss_df_output,trench_estimator_preinduction_df_output])
del trench_estimator_ss_df_output
del trench_estimator_preinduction_df_output
trench_estimator_df.to_pickle(trench_estimator_path)
del trench_estimator_df
trench_estimator_df = pd.read_pickle(trench_estimator_path)
trench_estimator_df = trench_estimator_df.reset_index()
trench_estimator_dd = dd.from_pandas(trench_estimator_df,100).persist();
trench_estimator_dd = trench_estimator_dd.set_index("Multi-Experiment Phenotype Trenchid").persist()
wait(trench_estimator_dd);
del trench_estimator_df

estimator_df = get_estimator_df(trench_estimator_dd,final_columns,param_groups_to_agg_list,estimator_names_to_agg,aggregators,is_paired_aggregetor_list,agg_names,sgrna_key='oDEPool7_id')
###Adding derived values
### note that CV (external) is normalized by wt median

'''
LUIS: REMOVED BECAUSE THERE ARE NO CONTROLS
'''
'''
control_estimator_df = estimator_df[estimator_df["Category"].isin(['OnlyPlasmid', 'NoTarget'])]
control_medians = control_estimator_df.loc["Mean (Robust)"].groupby("Variable(s)")["Value"].median()

CV_extrinsic_df = estimator_df.loc["Variance (Extrinsic)"].copy()
CV_extrinsic_df["Value"] = np.sqrt(estimator_df.loc["Variance (Extrinsic)"]["Value"])/control_medians
CV_extrinsic_df["Estimator"] = "CV (Extrinsic)"
CV_extrinsic_df = CV_extrinsic_df.reset_index().set_index(["Estimator","Variable(s)","oDEPool7_id"])
'''
'''
'''
# estimator_df_output = pd.concat([estimator_df,CV_extrinsic_df]).sort_index() #DANIEL'S VERSION
estimator_df_output = estimator_df.sort_index() # LUIS' VERSION
estimator_df_output.to_pickle(estimator_path)

In [ ]:
# Here ends main code

#### Growth Estimates

In [ ]:
single_variable_list = ['Instantaneous Growth Rate: major_axis_length', 'Instantaneous Growth Rate: Volume']

wrapped_single_variable_list = [[variable] for variable in single_variable_list]
param_groups_list=[wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list]
param_groups_to_agg_list=[wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                    wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                    wrapped_single_variable_list,wrapped_single_variable_list,wrapped_single_variable_list,\
                    wrapped_single_variable_list]

steady_state_df_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Growth_df"
preinduction_df_path = "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Preinduction_Growth_df"
trench_estimator_path = steady_state_df_path + "_Trench_Estimators.pkl"
estimator_path = steady_state_df_path + "_Estimators.pkl"

final_timepoints_df_ss = dd.read_parquet(steady_state_df_path, engine="pyarrow",calculate_divisions=True)
final_timepoints_df_preinduction = dd.read_parquet(preinduction_df_path, engine="pyarrow",calculate_divisions=True)

trench_estimator_ss_df_output = get_estimator_df_all_trenches(final_timepoints_df_ss,estimator_names,estimators,param_groups_list,final_columns,\
                                                              trench_index="Multi-Experiment Phenotype Trenchid")
trench_estimator_preinduction_df_output = get_estimator_df_all_trenches(final_timepoints_df_preinduction,estimator_names,estimators,param_groups_list,\
                                                                        final_columns,trench_index="Multi-Experiment Phenotype Trenchid")
trench_estimator_ss_df_output["Induction"] = "Post"
trench_estimator_preinduction_df_output["Induction"] = "Pre"
trench_estimator_ss_df_output = trench_estimator_ss_df_output.reset_index().set_index(["Induction","Estimator","Multi-Experiment Phenotype Trenchid"])
trench_estimator_preinduction_df_output = trench_estimator_preinduction_df_output.reset_index().set_index(["Induction","Estimator","Multi-Experiment Phenotype Trenchid"])

trench_estimator_df = pd.concat([trench_estimator_ss_df_output,trench_estimator_preinduction_df_output])
del trench_estimator_ss_df_output
del trench_estimator_preinduction_df_output
trench_estimator_df.to_pickle(trench_estimator_path)
del trench_estimator_df
trench_estimator_df = pd.read_pickle(trench_estimator_path)
trench_estimator_df = trench_estimator_df.reset_index()
trench_estimator_dd = dd.from_pandas(trench_estimator_df,100).persist();
trench_estimator_dd = trench_estimator_dd.set_index("Multi-Experiment Phenotype Trenchid").persist()
wait(trench_estimator_dd);
del trench_estimator_df

estimator_df = get_estimator_df(trench_estimator_dd,final_columns,param_groups_to_agg_list,estimator_names_to_agg,aggregators,is_paired_aggregetor_list,agg_names,sgrna_key='oDEPool7_id')
###Adding derived values
### note that CV (external) is normalized by wt median

'''
LUIS: REMOVED BECAUSE THERE ARE NO CONTROLS
'''
'''
control_estimator_df = estimator_df[estimator_df["Category"].isin(['OnlyPlasmid', 'NoTarget'])]
control_medians = control_estimator_df.loc["Mean (Robust)"].groupby("Variable(s)")["Value"].median()

CV_extrinsic_df = estimator_df.loc["Variance (Extrinsic)"].copy()
CV_extrinsic_df["Value"] = np.sqrt(estimator_df.loc["Variance (Extrinsic)"]["Value"])/control_medians
CV_extrinsic_df["Estimator"] = "CV (Extrinsic)"
CV_extrinsic_df = CV_extrinsic_df.reset_index().set_index(["Estimator","Variable(s)","oDEPool7_id"])
'''
'''
'''

# estimator_df_output = pd.concat([estimator_df,CV_extrinsic_df]).sort_index() # DANIEL'S VERSION
estimator_df_output = estimator_df.sort_index() # LUIS' VERSION
estimator_df_output.to_pickle(estimator_path)

In [ ]:
pd.read_pickle("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Timepoints_df_Trench_Estimators.pkl")

In [ ]:
df_trench = pd.read_pickle("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Timepoints_df_Trench_Estimators.pkl")

In [ ]:
df_agg = pd.read_pickle("/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Timepoints_df_Estimators.pkl")

In [ ]:
df_trench_pre = df_trench.loc['Pre', 'Mean']
df_trench_post = df_trench.loc["Post", 'Mean']

In [ ]:
df_trench_pre

In [ ]:
df_trench_post

In [ ]:
plt.hist(df_trench_pre['major_axis_length'], bins = 50, alpha = 0.3, density = True)
plt.hist(df_trench_post['major_axis_length'], bins = 100, alpha = 0.3, density= True)
plt.hist(df_trench_post_fts['major_axis_length'], bins = 100, alpha = 0.3, density= True)
plt.hist(df_trench_pre_fts['major_axis_length'], bins = 100, alpha = 0.3, density= True)
plt.xlim([0,15])

In [ ]:
plt.hist(df_trench_pre_fts['major_axis_length'], bins = 100, alpha = 0.3, density= True)
plt.hist(df_trench_post_fts['major_axis_length'], bins = 100, alpha = 0.3, density= True)
plt.xlim([0,15])

In [ ]:
df_trench_post_fts = df_trench_post[df_trench_post['Gene'].str.contains('fts')]

In [ ]:
df_trench_pre_fts = df_trench_pre[df_trench_pre['Gene'].str.contains('fts')]

In [ ]:
df_trench_luis = pd.read_pickle(
    "/home/lag36/scratch/lag36/2022-12-21_lDE25_Run_1/2023-01-03_lDE25_Steady_State_Timepoints_df_Trench_Estimators.pkl")

In [ ]:
df_trench_daniel = pd.read_pickle(
    "/home/de64/scratch/de64/sync_folder/2022-08-10_lDE23_LowSampling_Merged_Analysis/2022-09-22_lDE23_Steady_State_Timepoints_df_Trench_Estimators.pkl")

In [ ]:
df_trench_daniel["N Observations

In [ ]:
df_trench_daniel.loc["Post", 'Mean']

In [ ]:
df_trench_luis.loc["Post", 'Mean']

In [ ]:
df_trench_pre = df_trench.loc['Pre', 'Mean']
df_trench_post = df_trench.loc["Post", 'Mean']

In [ ]:

df_trench_post

In [ ]:
plt.plot(df_trench_post['major_axis_length'], df_trench_post['Total Nucleoid Area'], 'o', alpha = 0.05)
plt.plot(df_trench_pre['major_axis_length'], df_trench_pre['Total Nucleoid Area'], 'o', alpha = 0.05)

In [ ]:
plt.scatter(estimator_df.loc["Mean (Robust)","major_axis_length"]["Value"],\
            estimator_df.loc["Mean (Robust)","Total Nucleoid Area"]["Value"],s=3)

In [ ]:
ribostalk_df = estimator_df[estimator_df["Gene"].isin(["rplL","rplJ"])]

In [ ]:
estima

In [ ]:
plt.scatter(estimator_df.loc["Mean (Robust)","Number of Nucleoids"]["Value"],\
            estimator_df.loc["Mean (Robust)","Total Nucleoid Area"]["Value"],s=3)

plt.scatter(ribostalk_df.loc["Mean (Robust)","Number of Nucleoids"]["Value"],\
            ribostalk_df.loc["Mean (Robust)","Total Nucleoid Area"]["Value"],s=3)

In [ ]:
estimator_df.loc["Mean (Robust)","Number of Nucleoids"][estimator_df.loc["Mean (Robust)","Number of Nucleoids"]["Value"]>3]

In [ ]:
plt.plot(np.linspace(0,10,10),np.linspace(0,10,10))
estimator_df.loc["Mean (Robust)","Number of Nucleoids"][estimator_df.loc["Mean (Robust)","Total Nucleoid Area"]["Value"]>6]